# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [10]:
!pip install -qU datasets pinecone-client==3.1.0 sentence-transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 34.0 MB/s eta 0:00:00


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [2]:
!pip uninstall datasets -y
!pip install datasets==3.6.0


Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.9 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
wiki_data = load_dataset( 'vblagoje/wikipedia_snippets_streamed', split='train', streaming=True, trust_remote_code=True ).shuffle(seed=960)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


wikipedia_snippets_streamed.py: 0.00B [00:00, ?B/s]

In [ ]:
# # load the dataset from huggingface in streaming mode and shuffle it
# wiki_data = load_dataset(
#     'vblagoje/wikipedia_snippets_streamed',
#     split='train',
#     streaming=True
# ).shuffle(seed=960)

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [4]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'wiki_id': 'Q7649565',
 'start_paragraph': 20,
 'start_character': 272,
 'end_paragraph': 24,
 'end_character': 380,
 'article_title': 'Sustainable Agriculture Research and Education',
 'section_title': "2000s & Evaluation of the program's effectiveness",
 'passage_text': "preserving the surrounding prairies. It ran until March 31, 2001.\nIn 2008, SARE celebrated its 20th anniversary. To that date, the program had funded 3,700 projects and was operating with an annual budget of approximately $19 million. Evaluation of the program's effectiveness As of 2008, 64% of farmers who had received SARE grants stated that they had been able to earn increased profits as a result of the funding they received and utilization of sustainable agriculture methods. Additionally, 79% of grantees said that they had experienced a significant improvement in soil quality though the environmentally friendly, sustainable methods that they were"}

In [5]:
# filter only documents with History as section_title - Replace None with your code
history = wiki_data.filter(lambda x: "History" in x["section_title"])

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [6]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 5000

counter = 0
docs = []
# iterate through the dataset and apply our filter
for d in tqdm(history, total=total_doc_count):
    # extract the fields we need - article, section, and passage
    if counter >= total_doc_count:
        break
    docs.append({
    "article_title": d["article_title"],
    "section_title": d["section_title"],
    "passage_text": d["passage_text"]
    })
    # increase the counter on every iteration
    counter += 1

  0%|          | 0/5000 [00:00<?, ?it/s]

In [7]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Taupo District,History,was not until the 1950s that the region starte...
1,Sutarfeni,History & Western asian analogues,Sutarfeni History strand-like pheni were Phena...
2,The Bishop Wand Church of England School,History,The Bishop Wand Church of England School Histo...
3,Teufelsmoor,History & Situation today,"made to preserve the original landscape, altho..."
4,Surface Hill Uniting Church,History,in perpetual reminder that work and worship go...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [8]:
import os

# Ask user to input the API key securely
from getpass import getpass
os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API key: ")

# Access it when needed
pinecone_api_key = os.getenv("PINECONE_API_KEY")

Enter your Pinecone API key: ··········


In [14]:
import os
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.environ.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [11]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [12]:
index_name = "abstractive-question-answering"

In [22]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=spec   # Use the spec argument instead of pod_type
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

Index 'abstractive-question-answering' created.


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [23]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer("flax-sentence-embeddings/all_datasets_v3_mpnet-base")
retriever

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [28]:
index = pc.Index(index_name)

In [42]:
# we will use batches of 64
batch_size = 64

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    batch_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:batch_end]
    # generate embeddings for batch
    emb = retriever.encode(batch["passage_text"].tolist()).tolist()

    # Create unique string IDs from dataframe index
    ids = batch.index.astype(str).tolist()

    # Use other columns as metadata
    metadata = batch.to_dict(orient="records")

    # Upsert vectors into Pinecone index
    index.upsert(
        vectors=zip(ids, emb, metadata)
    )

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/79 [00:00<?, ?it/s]

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 5000}},
 'total_vector_count': 5000}

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [43]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [44]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode([query]).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq[0], top_k=top_k, include_metadata=True)
    return xc

In [45]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context_passages = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context_str = " ".join(context_passages)
    # contcatinate the query and context passages
    query_str = f"question: {query} context: {context_str}"
    return query_str

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [46]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

{'matches': [{'id': '3906',
              'metadata': {'article_title': 'Electric power system',
                           'passage_text': 'Electric power system History In '
                                           '1881, two electricians built the '
                                           "world's first power system at "
                                           'Godalming in England. It was '
                                           'powered by two waterwheels and '
                                           'produced an alternating current '
                                           'that in turn supplied seven '
                                           'Siemens arc lamps at 250 volts and '
                                           '34 incandescent lamps at 40 volts. '
                                           'However, supply to the lamps was '
                                           'intermittent and in 1882 Thomas '
                                           'Ed

In [48]:
from pprint import pprint

In [49]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

('question: question: when was the first electric power system built? context: '
 '<P> Electric power system History In 1881, two electricians built the '
 "world's first power system at Godalming in England. It was powered by two "
 'waterwheels and produced an alternating current that in turn supplied seven '
 'Siemens arc lamps at 250 volts and 34 incandescent lamps at 40 volts. '
 'However, supply to the lamps was intermittent and in 1882 Thomas Edison and '
 'his company, The Edison Electric Light Company, developed the first '
 'steam-powered electric power station on Pearl Street in New York City. The '
 'Pearl Street Station initially powered around 3,000 lamps for 59 customers. '
 'The power station generated direct current and context: <P> Electric power '
 "system History In 1881, two electricians built the world's first power "
 'system at Godalming in England. It was powered by two waterwheels and '
 'produced an alternating current that in turn supplied seven Siemens arc 

The output looks great. Now let's write a function to generate answers.

In [50]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [53]:
generate_answer(query)

('The first electric power system was built at Godalming in England in 1881. '
 'It was powered by two waterwheels and produced an alternating current that '
 'in turn supplied seven Siemens arc')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [54]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by a radio transmitter. The first radio '
 'transmitter was built in the early 1900s. The first radio transmitter was '
 'built in the early 1900s. The first radio')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [55]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

communications arose in the mid-1990s, when messaging and real-time communications began to combine. In 1993, ThinkRite (VoiceRite) developed the unified messaging system, POET, for IBM's internal use. It was installed in 55 IBM US Branch Offices for 54,000 employees and integrated with IBM OfficeVision/VM (PROFS) and provided IBMers with one phone number for voicemail, fax, alphanumeric paging and follow-me.  POET was in use until 2000.  In the late 1990s, a New Zealand-based organization called IPFX developed a commercially available presence product, which let users see the location of colleagues, make decisions on how to contact them, and define
---
introduced from 1988 onwards, and the JPEG image compression standard introduced in 1992. Experiments There were several early videophones and cameras that included communication capability. Some devices experimented with integration of the device to communicate wirelessly with the Internet, which would allow instant media sharing with 

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [56]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is a strain of the coronaviruses. The coronaviruses are a group of '
 'viruses that infect humans. The coronaviruses are a group of viruses')


In [57]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

1974 during the Super Outbreak (one of seven F5 tornadoes during that outbreak) which killed three and demolished many homes.
---
high enough to meet the demands of global health. The GVN also receives funding from its partners, the Global Health Security Security Agenda (GHSA), the Coalition for Epidemic Preparedness Innovations (CEPI), and the Medical Technology Enterprise Consortium (MTEC). History and centers The GVN has 34 centers across 17 nations, each of which has a medical virologist as its director who is willing to commit to GVN's mission. Each center also specializes in at least two areas of viral science. Chikungunya virus explained CHIKV is a rapid-working onset febrile illness, of which the first visible symptom is a rash on the
---
from South Arabia. Karibi-ilu built a house or a temple called "Bit-Akitu", to celebrate the New Year's and other holidays.
---


Let’s finish with a final few questions.

In [58]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The War of Currents was a series of events in the early 20th century that '
 'led to the development of alternating current power. The first major power '
 'plant was built by Westinghouse in')


In [60]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

('The first person to walk on the moon was Neil Armstrong in 1969. He was the '
 'first man to walk on the moon, but he was not the first person to walk on '
 'the moon.')


In [61]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of the US '
 'government. It cost about $400 billion to build.')


As we can see, the model can generate some decent answers.

#### Add a few more questions

In [62]:
query = "what was the reason for WW1 to start?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

("I'm going to assume you're talking about the First World War. The war "
 'started in 1914 when Germany invaded Austria-Hungary. The Austro-Hungarian '
 'Empire had been invaded by')


In [63]:
query = "which civilization was the first to make gunpowder?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

("I don't know about gunpowder, but I can tell you about gunpowder in general. "
 'Gunpowder is a type of explosive that is used to propel a projectile. It is '
 'used in')


In [64]:
query = "when was the american civil war?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

('The Civil War is a very broad term. It covers a wide range of events, from '
 'the founding of the United States to the end of the American Civil War. It '
 'is not a single')
